In [ ]:
from utils import ModelConfig, SwiGLUBlock

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoE(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.num_experts = config.num_experts
        self.top_k = config.moe_router_topk
        
        # Router: 根据要求不使用 bias
        self.router = nn.Linear(config.hidden_size, config.num_experts, bias=False)
        
        # 专家池
        self.experts = nn.ModuleList([MLP(config) for _ in range(config.num_experts)])

    def _compute_aux_loss(self, router_probs, indices):
        """
        计算辅助损失 (Auxiliary Loss)
        router_probs: [num_tokens, num_experts] 归一化后的概率
        indices: [num_tokens, top_k] 选中的专家索引
        """
        
        # 计算每个专家的负载 (Fraction of tokens assigned to each expert)
        # 统计 top-1 选中的频率 (Megatron 通常基于 top-1 计算负载)
        top1_indices = indices[:, 0]
        expert_mask = F.one_hot(top1_indices, num_experts=self.num_experts).float()
        
        # f_i: 选该专家的 token 占比
        f = expert_mask.mean(dim=0) 
        
        # P_i: 专家收到的总概率均值
        P = router_probs.mean(dim=0)
        
        # 根据配置返回不同类型的损失
        if self.config.moe_router_load_balancing_type == "aux_loss":
            # 标准负载均衡损失: L = N * sum(f_i * P_i)
            loss = self.num_experts * torch.sum(f * P)
        elif self.config.moe_router_load_balancing_type == "seq_aux_loss":
            # 序列级的辅助损失 (此处简化实现其逻辑等效性)
            loss = torch.sum(f * P) * self.config.moe_aux_loss_coeff
        else:
            loss = torch.tensor(0.0, device=router_probs.device)
            
        return loss

    def forward(self, hidden_states):
        # hidden_states: [b, s, h] -> [tokens, h]
        orig_shape = hidden_states.shape
        x = hidden_states.view(-1, self.config.hidden_size)

        # 1. 计算路由分数
        logits = self.router(x)
        
        # 2. 获取 Top-K
        # 在计算 loss 时需要 full probs
        all_probs = F.softmax(logits, dim=-1) 
        weights, selected_experts = torch.topk(logits, self.top_k, dim=-1)
        routing_weights = F.softmax(weights, dim=-1) # [num_tokens, top_k]

        # 3. 计算辅助损失
        aux_loss = self._compute_aux_loss(all_probs, selected_experts)

        # 4. 专家执行 (最简循环实现)
        final_output = torch.zeros_like(x)
        
        # 模拟 Token 分发
        for k in range(self.top_k):
            expert_idx_for_k = selected_experts[:, k]
            weight_for_k = routing_weights[:, k].unsqueeze(-1)
            
            for i in range(self.num_experts):
                mask = (expert_idx_for_k == i)
                if mask.any():
                    token_indices = mask.nonzero(as_tuple=True)[0]
                    out = self.experts[i](x[token_indices])
                    final_output[token_indices] += weight_for_k[token_indices] * out

        return final_output.view(orig_shape), aux_loss

In [ ]:
from config import ModelArgs

model_args = ModelArgs()

print(">>> 正在使用 ModelArgs 初始化 StandardTopKMoELayer...")
moe_layer = TopKMoELayer(model_args)

# 验证关键参数
print(f"Experts Count: {len(moe_layer.experts)} (Config: 64)")
print(f"Top K: {moe_layer.top_k} (Config: 4)")
print(f"Expert Hidden Dim: {moe_layer.experts[0].gate_proj.out_features} (Config: 768)")
print(f"Router Bias Enabled: {moe_layer.router.bias is not None} (Config: True)")
print(f"Router Score Function: {model_args.moe_router_score_function} (Config: sigmoid)")

# 运行 Dummy Forward
x = torch.randn(2, 128, 2048) # (Batch, Seq, Hidden)
output, aux_loss = moe_layer(x)

print("\n>>> Forward Pass Successful")
print(f"Input Shape: {x.shape}")
print(f"Output Shape: {output.shape}")
print(f"Aux Loss: {aux_loss}")